# DistilBERT (Colab) - Training-Size Benchmark for Jigsaw

Colab-ready notebook based on `04_distilbert.ipynb`.

It benchmarks DistilBERT over increasing training sizes (step 10,000), uses 2 epochs per size,
applies per-label `pos_weight`, tunes thresholds, and saves CSV artifacts to Google Drive.

In [ ]:
# Colab setup: install dependencies
!pip -q install torch transformers pandas numpy matplotlib scikit-learn

In [ ]:
# Mount Google Drive and set paths
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ===== CHANGE THIS to your repo folder in Drive =====
PROJECT_ROOT = Path('/content/drive/MyDrive/cmpe258/jigsaw-toxic-comment-classification-challenge')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'PROJECT_ROOT not found: {PROJECT_ROOT}. Upload/clone your repo to this path first.'
    )

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
ARTIFACT_DIR = NOTEBOOKS_DIR / 'artifacts_colab'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOKS_DIR))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_distilbert
from metrics_helpers import multilabel_evaluation_report, torch_parameter_count

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def binary_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else 0.0


def tune_per_label_thresholds(y_true: np.ndarray, y_prob: np.ndarray, labels, threshold_grid: np.ndarray):
    best_thresholds = {}
    rows = []
    for j, label in enumerate(labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]
        best_t = 0.5
        best_f1 = -1.0
        for t in threshold_grid:
            y_pred_j = (y_prob_j >= t).astype(np.int64)
            f1_j = binary_f1(y_true_j, y_pred_j)
            if f1_j > best_f1:
                best_f1 = f1_j
                best_t = float(t)
        best_thresholds[label] = best_t
        rows.append({'label': label, 'best_threshold': round(best_t, 3), 'best_f1_on_val': best_f1})
    return best_thresholds, pd.DataFrame(rows)


def apply_thresholds(y_prob: np.ndarray, labels, thresholds: dict[str, float]) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=np.int64)
    for j, label in enumerate(labels):
        out[:, j] = (y_prob[:, j] >= thresholds[label]).astype(np.int64)
    return out


def compute_pos_weight(y_train: np.ndarray, eps: float = 1e-6, max_weight: float = 50.0) -> torch.Tensor:
    y = np.asarray(y_train, dtype=np.float32)
    positives = y.sum(axis=0)
    total = float(y.shape[0])
    negatives = total - positives
    weights = negatives / np.maximum(positives, eps)
    weights = np.clip(weights, 1.0, max_weight)
    return torch.tensor(weights, dtype=torch.float32)


def enc_dict(enc):
    keys = [k for k in ('input_ids', 'attention_mask') if k in enc]
    return {k: enc[k] for k in keys}


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != 'labels'}
    out['labels'] = torch.stack([b['labels'] for b in batch], dim=0)
    return out

In [ ]:
# Benchmark configuration
DEVICE = pick_device()
torch.manual_seed(42)
np.random.seed(42)

MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
SIZE_STEP = 10000
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)
MAX_TRAIN_CAP = None      # set int to cap max train rows
MAX_VAL_SAMPLES = None    # set int for faster debug
SKIP_COMPLETED = True

probe = preprocess_for_distilbert(
    validation_fraction=0.1,
    random_state=42,
    max_length=MAX_LENGTH,
    return_tensors='pt',
    max_train_samples=MAX_TRAIN_CAP,
    max_val_samples=MAX_VAL_SAMPLES,
)
max_train_available = len(probe.y_train)

if max_train_available < SIZE_STEP:
    train_sizes = [max_train_available]
else:
    train_sizes = list(range(SIZE_STEP, max_train_available + 1, SIZE_STEP))
    if train_sizes[-1] != max_train_available:
        train_sizes.append(max_train_available)

print('Device:', DEVICE)
print('max_train_available:', max_train_available)
print('train_sizes:', train_sizes[:5], '...', train_sizes[-1], f'(n={len(train_sizes)})')

In [ ]:
# Run benchmark sweep
summary_rows = []
per_label_frames = []
threshold_frames = []

for idx, train_size in enumerate(train_sizes, start=1):
    row_file = ARTIFACT_DIR / f'summary_{train_size}.csv'
    per_label_file = ARTIFACT_DIR / f'per_label_{train_size}.csv'
    thresholds_file = ARTIFACT_DIR / f'thresholds_{train_size}.csv'

    if SKIP_COMPLETED and row_file.exists() and per_label_file.exists() and thresholds_file.exists():
        print(f'[{idx}/{len(train_sizes)}] train_size={train_size}: skipping (already exists)')
        summary_rows.append(pd.read_csv(row_file).iloc[0].to_dict())
        per_label_frames.append(pd.read_csv(per_label_file))
        threshold_frames.append(pd.read_csv(thresholds_file))
        continue

    print(f'[{idx}/{len(train_sizes)}] train_size={train_size}: running')
    run_data = preprocess_for_distilbert(
        validation_fraction=0.1,
        random_state=42,
        max_length=MAX_LENGTH,
        return_tensors='pt',
        max_train_samples=train_size,
        max_val_samples=MAX_VAL_SAMPLES,
    )

    train_enc = enc_dict(run_data.train_encodings)
    val_enc = enc_dict(run_data.val_encodings)
    y_train = np.asarray(run_data.y_train, dtype=np.float32)
    y_val = np.asarray(run_data.y_val, dtype=np.int64)

    train_loader = DataLoader(EncDataset(train_enc, y_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(EncDataset(val_enc, y_val), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

    model = AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased',
        num_labels=len(LABEL_COLUMNS),
        problem_type='multi_label_classification',
    ).to(DEVICE)
    num_params = torch_parameter_count(model)

    pos_weight = compute_pos_weight(y_train).to(DEVICE)
    pos_weight_dict = {label: float(pos_weight[j].item()) for j, label in enumerate(LABEL_COLUMNS)}

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    train_time_s = 0.0
    inference_time_s = 0.0
    train_loss = np.nan
    val_loss = np.nan

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_train_loss = 0.0
        epoch_batches = 0
        t0 = time.perf_counter()
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop('labels')
            optimizer.zero_grad()
            logits = model(**batch).logits
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            epoch_train_loss += float(loss.item())
            epoch_batches += 1
        train_time_s += time.perf_counter() - t0
        train_loss = epoch_train_loss / max(epoch_batches, 1)

        model.eval()
        probs_all = []
        val_loss_sum = 0.0
        val_batches = 0
        t1 = time.perf_counter()
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                labels = batch.pop('labels')
                logits = model(**batch).logits
                probs = torch.sigmoid(logits)
                probs_all.append(probs.cpu().numpy())
                val_loss_sum += float(loss_fn(logits, labels).item())
                val_batches += 1
        inference_time_s += time.perf_counter() - t1
        val_loss = val_loss_sum / max(val_batches, 1)

    prob_val = np.concatenate(probs_all, axis=0)
    pred_05 = (prob_val >= 0.5).astype(np.int64)
    per_label_05_df, summary_05 = multilabel_evaluation_report(y_val, pred_05, prob_val, LABEL_COLUMNS)

    best_thresholds, thresholds_df = tune_per_label_thresholds(y_val, prob_val, LABEL_COLUMNS, THRESHOLD_GRID)
    pred_tuned = apply_thresholds(prob_val, LABEL_COLUMNS, best_thresholds)
    per_label_tuned_df, summary_tuned = multilabel_evaluation_report(y_val, pred_tuned, prob_val, LABEL_COLUMNS)

    per_label_df = per_label_05_df.rename(
        columns={
            'precision': 'precision_baseline',
            'recall': 'recall_baseline',
            'f1': 'f1_baseline',
            'roc_auc': 'roc_auc_baseline',
        }
    ).merge(
        per_label_tuned_df.rename(
            columns={
                'precision': 'precision_tuned',
                'recall': 'recall_tuned',
                'f1': 'f1_tuned',
                'roc_auc': 'roc_auc_tuned',
            }
        ),
        on='label',
        how='left',
    )
    per_label_df['train_size'] = train_size
    for label in LABEL_COLUMNS:
        per_label_df[f'pos_weight_{label}'] = pos_weight_dict[label]

    thresholds_df['train_size'] = train_size

    row = {
        'train_size': train_size,
        'epochs': EPOCHS,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'num_parameters': num_params,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_time_s': train_time_s,
        'inference_time_s': inference_time_s,
        'f1_micro_baseline': summary_05['f1_micro'],
        'f1_macro_baseline': summary_05['f1_macro'],
        'f1_micro_tuned': summary_tuned['f1_micro'],
        'f1_macro_tuned': summary_tuned['f1_macro'],
        'pos_weight_min': float(pos_weight.min().item()),
        'pos_weight_max': float(pos_weight.max().item()),
        'pos_weight_mean': float(pos_weight.mean().item()),
    }

    pd.DataFrame([row]).to_csv(row_file, index=False)
    per_label_df.to_csv(per_label_file, index=False)
    thresholds_df.to_csv(thresholds_file, index=False)

    summary_rows.append(row)
    per_label_frames.append(per_label_df)
    threshold_frames.append(thresholds_df)

benchmark_summary_df = pd.DataFrame(summary_rows).sort_values('train_size').reset_index(drop=True)
benchmark_per_label_df = pd.concat(per_label_frames, ignore_index=True)
benchmark_thresholds_df = pd.concat(threshold_frames, ignore_index=True)

benchmark_summary_df['f1_micro_delta_tuned_minus_baseline'] = (
    benchmark_summary_df['f1_micro_tuned'] - benchmark_summary_df['f1_micro_baseline']
)
benchmark_summary_df['f1_macro_delta_tuned_minus_baseline'] = (
    benchmark_summary_df['f1_macro_tuned'] - benchmark_summary_df['f1_macro_baseline']
)

summary_csv = ARTIFACT_DIR / 'distilbert_size_benchmark_summary.csv'
per_label_csv = ARTIFACT_DIR / 'distilbert_size_benchmark_per_label.csv'
thresholds_csv = ARTIFACT_DIR / 'distilbert_size_benchmark_thresholds.csv'

benchmark_summary_df.to_csv(summary_csv, index=False)
benchmark_per_label_df.to_csv(per_label_csv, index=False)
benchmark_thresholds_df.to_csv(thresholds_csv, index=False)

print('Saved:')
print(' ', summary_csv)
print(' ', per_label_csv)
print(' ', thresholds_csv)
display(benchmark_summary_df)

In [ ]:
# Plots: metric changes vs training size
plt.figure(figsize=(8, 4.5))
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['train_loss'], marker='o', label='train_loss')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['val_loss'], marker='o', label='val_loss')
plt.title('Loss vs Train Size')
plt.xlabel('Train size')
plt.ylabel('Loss')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['f1_micro_baseline'], marker='o', label='micro F1 (0.5)')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['f1_macro_baseline'], marker='o', label='macro F1 (0.5)')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['f1_micro_tuned'], marker='o', linestyle='--', label='micro F1 (tuned)')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['f1_macro_tuned'], marker='o', linestyle='--', label='macro F1 (tuned)')
plt.title('Aggregate F1 vs Train Size')
plt.xlabel('Train size')
plt.ylabel('F1')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['train_time_s'], marker='o', label='train_time_s')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['inference_time_s'], marker='o', label='inference_time_s')
plt.title('Runtime vs Train Size')
plt.xlabel('Train size')
plt.ylabel('Seconds')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5.5))
for label in LABEL_COLUMNS:
    df_l = benchmark_per_label_df.loc[benchmark_per_label_df['label'] == label].sort_values('train_size')
    plt.plot(df_l['train_size'], df_l['f1_tuned'], marker='o', label=f'{label} (tuned)')
plt.title('Per-label Tuned F1 vs Train Size')
plt.xlabel('Train size')
plt.ylabel('F1')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend(loc='best')
plt.show()